#Reading From Bronze Table

# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DataType, DoubleType, IntegerType
from pyspark.sql.functions import trim, col
from pyspark.sql.functions import split, col
from pyspark.sql.window import Window

%md
## erp_loc_a101 Transformation

In [0]:
# ============================================================
# ERP_LOC_A101 — Customer Location / Country
# ============================================================
df_loc = spark.table("dev_project.bronze.erp_loc_a101")

# Trim all string columns
string_cols = [f.name for f in df_loc.schema.fields if isinstance(f.dataType, StringType)]
for col_name in string_cols:
    df_loc = df_loc.withColumn(
        col_name,
        F.when(F.trim(F.col(col_name)) == "", None)
         .otherwise(F.trim(F.col(col_name)))
    )

# Clean CID — remove hyphens to get AW00011000 format
df_loc = df_loc.withColumn(
    "CID",
    F.regexp_replace(F.col("CID"), "-", "")
)

# Normalize CNTRY — standardize all country variants
df_loc = df_loc.withColumn(
    "CNTRY",
    F.when(F.upper(F.trim(F.col("CNTRY"))).isin("US", "USA", "UNITED STATES"), "United States")
     .when(F.upper(F.trim(F.col("CNTRY"))).isin("DE", "GERMANY"), "Germany")
     .when(F.upper(F.trim(F.col("CNTRY"))) == "FRANCE", "France")
     .when(F.upper(F.trim(F.col("CNTRY"))) == "AUSTRALIA", "Australia")
     .when(F.upper(F.trim(F.col("CNTRY"))) == "CANADA", "Canada")
     .when(F.upper(F.trim(F.col("CNTRY"))) == "UNITED KINGDOM", "United Kingdom")
     .when(F.col("CNTRY").isNull(), "n/a")
     .otherwise("n/a")
)

# Rename + final select
df_loc_silver = df_loc.select(
    F.col("CID").alias("customer_key"),
    F.col("CNTRY").alias("country")
)

# Validation
print("\n" + "="*50)
print(" ERP_LOC_A101 VALIDATION")
print("="*50)
print(f"Total records:      {df_loc_silver.count():,}")
print(f"Null customer keys: {df_loc_silver.filter(F.col('customer_key').isNull()).count()}")
print(f"Country breakdown:")
df_loc_silver.groupBy("country").count().orderBy("count", ascending=False).show()

# Write — overwrite (reference table, current state only)
(
    df_loc_silver.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("dev_project.silver.erp_loc_a101")
)
print("erp_loc_a101 Silver write complete.")